# CartIQ — Phase 3: Random Forest Baseline & GBDT
**CS Machine Learning Course Project — Group 3**

Trains Random Forest (baseline) and LightGBM (primary) on the engineered feature set. Includes Optuna hyperparameter tuning and SHAP explainability.

In [ ]:
!pip install -q lightgbm optuna shap scikit-learn matplotlib seaborn

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os, json, time, warnings
warnings.filterwarnings('ignore')

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    precision_score, recall_score, f1_score, roc_auc_score,
    classification_report, confusion_matrix, RocCurveDisplay
)
import lightgbm as lgb
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)
import shap

sns.set_style('whitegrid')
print('All libraries loaded.')

In [ ]:
# Load data
train_df = pd.read_parquet('data/train.parquet')
val_df = pd.read_parquet('data/val.parquet')
test_df = pd.read_parquet('data/test.parquet')
feature_cols = pd.read_csv('data/feature_cols.csv', header=None)[0].tolist()

X_train = train_df[feature_cols].values
y_train = train_df['label'].values
X_val = val_df[feature_cols].values
y_val = val_df['label'].values
X_test = test_df[feature_cols].values
y_test = test_df['label'].values

neg_count = (y_train == 0).sum()
pos_count = (y_train == 1).sum()
scale_pos_weight = neg_count / pos_count

print(f'Train: {X_train.shape} | Val: {X_val.shape} | Test: {X_test.shape}')
print(f'Features: {len(feature_cols)}')
print(f'Class imbalance — neg:pos = {scale_pos_weight:.2f}:1')

---
## 3.1 Random Forest Baseline

In [ ]:
print('Training Random Forest (200 trees)...')
t0 = time.time()

rf = RandomForestClassifier(
    n_estimators=200,
    max_depth=12,
    min_samples_leaf=50,
    class_weight='balanced',
    n_jobs=-1,
    random_state=42,
)
rf.fit(X_train, y_train)

rf_time = time.time() - t0
print(f'Trained in {rf_time:.1f}s')

In [ ]:
# Evaluate Random Forest
rf_proba = rf.predict_proba(X_test)[:, 1]
rf_pred = (rf_proba >= 0.5).astype(int)

rf_metrics = {
    'model': 'Random Forest',
    'precision': precision_score(y_test, rf_pred),
    'recall': recall_score(y_test, rf_pred),
    'f1': f1_score(y_test, rf_pred),
    'roc_auc': roc_auc_score(y_test, rf_proba),
}

print('=== Random Forest (Baseline) ===')
print(f'Precision: {rf_metrics["precision"]:.4f}')
print(f'Recall:    {rf_metrics["recall"]:.4f}')
print(f'F1-Score:  {rf_metrics["f1"]:.4f}')
print(f'ROC-AUC:   {rf_metrics["roc_auc"]:.4f}')
print()
print(classification_report(y_test, rf_pred, target_names=['Not Reordered', 'Reordered']))

In [ ]:
# RF Feature Importance
rf_importance = pd.Series(rf.feature_importances_, index=feature_cols).sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(10, 8))
rf_importance.head(15).plot(kind='barh', ax=ax, color='#3b82f6')
ax.invert_yaxis()
ax.set_title('Random Forest — Top 15 Feature Importances', fontweight='bold', fontsize=14)
ax.set_xlabel('Importance')
plt.tight_layout()
plt.savefig('rf_feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 3.2 LightGBM with Optuna Hyperparameter Tuning

In [ ]:
def objective(trial):
    params = {
        'objective': 'binary',
        'metric': 'binary_logloss',
        'boosting_type': 'gbdt',
        'scale_pos_weight': scale_pos_weight,
        'verbosity': -1,
        'n_jobs': -1,
        'seed': 42,
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'num_leaves': trial.suggest_int('num_leaves', 16, 128),
        'max_depth': trial.suggest_int('max_depth', 4, 12),
        'min_child_samples': trial.suggest_int('min_child_samples', 20, 200),
        'subsample': trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-8, 10.0, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-8, 10.0, log=True),
    }

    dtrain = lgb.Dataset(X_train, label=y_train)
    dval = lgb.Dataset(X_val, label=y_val, reference=dtrain)

    model = lgb.train(
        params, dtrain,
        num_boost_round=500,
        valid_sets=[dval],
        callbacks=[lgb.early_stopping(30, verbose=False)],
    )

    val_proba = model.predict(X_val)
    return f1_score(y_val, (val_proba >= 0.5).astype(int))

print('Running Optuna hyperparameter search (50 trials)...')
t0 = time.time()

study = optuna.create_study(direction='maximize', study_name='lgbm_cartiq')
study.optimize(objective, n_trials=50, show_progress_bar=True)

optuna_time = time.time() - t0
print(f'\nOptuna completed in {optuna_time:.1f}s')
print(f'Best F1: {study.best_value:.4f}')
print(f'Best params: {json.dumps(study.best_params, indent=2)}')

In [ ]:
# Train final GBDT with best params
best_params = {
    'objective': 'binary',
    'metric': 'binary_logloss',
    'boosting_type': 'gbdt',
    'scale_pos_weight': scale_pos_weight,
    'verbosity': -1,
    'n_jobs': -1,
    'seed': 42,
    **study.best_params,
}

dtrain = lgb.Dataset(X_train, label=y_train)
dval = lgb.Dataset(X_val, label=y_val, reference=dtrain)

print('Training final LightGBM model...')
gbdt = lgb.train(
    best_params, dtrain,
    num_boost_round=1000,
    valid_sets=[dval],
    callbacks=[lgb.early_stopping(50, verbose=True), lgb.log_evaluation(100)],
)

print(f'\nBest iteration: {gbdt.best_iteration}')

In [ ]:
# Evaluate GBDT
gbdt_proba = gbdt.predict(X_test)
gbdt_pred = (gbdt_proba >= 0.5).astype(int)

gbdt_metrics = {
    'model': 'LightGBM (GBDT)',
    'precision': precision_score(y_test, gbdt_pred),
    'recall': recall_score(y_test, gbdt_pred),
    'f1': f1_score(y_test, gbdt_pred),
    'roc_auc': roc_auc_score(y_test, gbdt_proba),
}

print('=== LightGBM (GBDT — Primary) ===')
print(f'Precision: {gbdt_metrics["precision"]:.4f}')
print(f'Recall:    {gbdt_metrics["recall"]:.4f}')
print(f'F1-Score:  {gbdt_metrics["f1"]:.4f}')
print(f'ROC-AUC:   {gbdt_metrics["roc_auc"]:.4f}')
print()
print(classification_report(y_test, gbdt_pred, target_names=['Not Reordered', 'Reordered']))

---
## 3.3 SHAP Explainability Analysis

In [ ]:
# SHAP analysis on a sample (full dataset is too large for SHAP)
sample_idx = np.random.RandomState(42).choice(len(X_test), size=min(5000, len(X_test)), replace=False)
X_sample = X_test[sample_idx]

explainer = shap.TreeExplainer(gbdt)
shap_values = explainer.shap_values(X_sample)

print(f'SHAP values computed for {len(X_sample)} samples.')

In [ ]:
# SHAP Summary Plot
fig, ax = plt.subplots(figsize=(12, 8))
shap.summary_plot(shap_values, X_sample, feature_names=feature_cols, show=False, max_display=15)
plt.title('SHAP Feature Importance — LightGBM', fontweight='bold', fontsize=14)
plt.tight_layout()
plt.savefig('shap_summary.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# SHAP Bar Plot (mean absolute SHAP values)
fig, ax = plt.subplots(figsize=(10, 8))
shap.summary_plot(shap_values, X_sample, feature_names=feature_cols,
                  plot_type='bar', show=False, max_display=15)
plt.title('Mean |SHAP| — Feature Impact on Reorder Prediction', fontweight='bold', fontsize=14)
plt.tight_layout()
plt.savefig('shap_bar.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 3.4 ROC Curves

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))

RocCurveDisplay.from_predictions(y_test, rf_proba, name='Random Forest', ax=ax, color='#3b82f6')
RocCurveDisplay.from_predictions(y_test, gbdt_proba, name='LightGBM', ax=ax, color='#ef4444')

ax.plot([0, 1], [0, 1], 'k--', alpha=0.3)
ax.set_title('ROC Curves — RF vs GBDT', fontweight='bold', fontsize=14)
ax.legend(fontsize=12)
plt.tight_layout()
plt.savefig('roc_rf_gbdt.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 3.5 Confusion Matrices

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, pred, name in [(axes[0], rf_pred, 'Random Forest'), (axes[1], gbdt_pred, 'LightGBM')]:
    cm = confusion_matrix(y_test, pred)
    sns.heatmap(cm, annot=True, fmt=',d', cmap='Blues', ax=ax,
                xticklabels=['Not Reordered', 'Reordered'],
                yticklabels=['Not Reordered', 'Reordered'])
    ax.set_title(f'{name}', fontweight='bold', fontsize=13)
    ax.set_xlabel('Predicted')
    ax.set_ylabel('Actual')

plt.tight_layout()
plt.savefig('confusion_rf_gbdt.png', dpi=150, bbox_inches='tight')
plt.show()

## 3.6 Save Models & Metrics

In [ ]:
import pickle

os.makedirs('models', exist_ok=True)

# Save RF
with open('models/random_forest.pkl', 'wb') as f:
    pickle.dump(rf, f)

# Save GBDT
gbdt.save_model('models/lightgbm.txt')

# Save metrics
metrics = [rf_metrics, gbdt_metrics]
pd.DataFrame(metrics).to_csv('models/metrics_traditional.csv', index=False)

# Save best params
with open('models/lgbm_best_params.json', 'w') as f:
    json.dump(study.best_params, f, indent=2)

print('Models saved:')
for f in os.listdir('models'):
    size = os.path.getsize(os.path.join('models', f)) / 1e6
    print(f'  models/{f:35s} {size:8.4f} MB')

print('\n=== Comparison So Far ===')
display(pd.DataFrame(metrics).set_index('model'))